In [1]:
from ngsolve import *
from ngsolve.webgui import Draw
from math import log as ln
from ngsolve.solvers import Newton

In [2]:
#generate mesh of beam
from netgen.occ import *
shape = Rectangle(1,0.1).Face()
shape.edges.Max(X).name="right"
shape.edges.Min(X).name="left"
shape.edges.Max(Y).name="top"
shape.edges.Min(Y).name="bot"
mesh = Mesh(OCCGeometry(shape, dim=2).GenerateMesh(maxh=0.03))

In [3]:
Draw(mesh)

WebGLScene

In [4]:
print(mesh.GetBoundaries)

<bound method pybind11_detail_function_record_v1_system_libstdcpp_gxx_abi_1xxx_use_cxx11_abi_1.GetBoundaries of <ngsolve.comp.Mesh object at 0x7fcefc0f67b0>>


In [5]:
#define finite element space (left boundary is fixed)
order = 2
fes = VectorH1(mesh, order=order, dirichlet="left")

In [6]:
#define bilinear form and right hand side to be gravity as volume force (E, nu are material parameters)
g = 9.81

#take density of steel for now and parameters such that we have significant deformation
width = 0.1
rho = 7850 #kg/m^3, converted to 2d density by formulation (cancels out with left hand side)
E, nu = 50e6, 0.3 #Pa, 1
mu  = E / 2 / (1+nu)
lam = E * nu / ((1+nu)*(1-2*nu))
def Stress(strain):
    return 2*mu*strain + lam*Trace(strain)*Id(2)

u,v = fes.TnT()
f = CoefficientFunction((0,-g))*rho
a = BilinearForm(InnerProduct(Stress(Sym(Grad(u))), Sym(Grad(v)))*dx)
lf = LinearForm(fes)
lf += f* v * dx

In [7]:
#assemble forms
with TaskManager():
    a.Assemble()
    lf.Assemble()

In [8]:
#solve (directly for now) without constraints
gfu = GridFunction(fes)
with TaskManager():
    gfu.vec.data = a.mat.Inverse(freedofs=fes.FreeDofs()) *lf.vec

In [9]:
#plot solution
Draw(gfu, mesh, deformation=True)

WebGLScene

In [10]:
#now in the constraint problem, we want to have a contact point on the right side,
#i.e, u_y >=0 on (0,7,1)x{0}

In [11]:
# Artificial floor on the left.
# This must only be lower than all expected displacements.
M = 10.0
H = 0.2
# Small strictly positive initial gap
eps0 = H+1e-3
# Lower obstacle for the vertical displacement:
#
#     u_y >= phi
#
# phi = H   on x > 0.75
# phi = -M  on x < 0.75
phi = IfPos(x - 0.75, H, -M)

In [12]:
#define space for the constraint
W = H1(
    mesh,
    order=order,
    definedon=mesh.Boundaries("bot"),
)

# Alternative for a sharper representation of the jump in phi:
#
# W = SurfaceL2(
#     mesh,
#     order=0,
#     definedon=mesh.Boundaries("bot"),
# )

In [13]:
# Product space: displacement in the volume, latent variable on bot
X = fes * W

u, psi = X.TrialFunction()
v, w = X.TestFunction()


gfcontact = GridFunction(X)
uh, psih = gfcontact.components


# Latent variable from the previous OUTER LVPP iteration
psi_old = GridFunction(W)

In [14]:
# Choose psi^0 such that
#
#     phi + exp(psi^0) = eps0
#
psi0 = IfPos(
    x - 0.75,
    ln(eps0),
    ln(M + eps0),
)

psi_old.Set(
    psi0,
    definedon=mesh.Boundaries("bot"),
)

# Initial guess for the first Newton solve
psih.vec.data = psi_old.vec


# Proximal parameter as a mutable CoefficientFunction
alpha = Parameter(5e-4)

In [15]:
F = BilinearForm(X)

# Mechanical equilibrium:
#
# alpha * (a(u,v) - l(v))
F += alpha * (
    InnerProduct(
        Stress(Sym(Grad(u))),
        Sym(Grad(v)),
    )
    - f * v
) * dx


# Proximal contact force:
#
# <psi - psi_old, v_y>_bot
F += (
    psi - psi_old
) * v[1] * ds("bot")


# Latent gap relation:
#
# u_y - phi = exp(psi)
F += (
    u[1]
    - phi
    - exp(psi)
) * w * ds("bot")

In [16]:
u_old = GridFunction(fes)
pressure = GridFunction(W)

alpha_k = 5e-5

with TaskManager():
    for k in range(20):

        # Store u^{k-1} for the outer stopping criterion
        u_old.vec.data = uh.vec

        alpha.Set(alpha_k)

        # Solve the current smooth LVPP subproblem
        Newton(
            F,
            gfcontact,
            inverse="umfpack",
            maxit=20,
            maxerr=1e-9,
            printing=False,
        )

        # Contact pressure:
        #
        # p^k = (psi^{k-1} - psi^k) / alpha_k
        #
        # This must be computed before psi_old is overwritten.
        pressure.vec.data = psi_old.vec
        pressure.vec.data -= psih.vec
        pressure.vec.data *= 1.0 / alpha_k

        # L2 change of the displacement between outer iterations
        du = sqrt(
            Integrate(
                InnerProduct(
                    uh - u_old,
                    uh - u_old,
                ),
                mesh,
            )
        )

        print(
            f"{k:2d}: "
            f"alpha = {alpha_k:8.3e}, "
            f"||u^k-u^(k-1)||_L2 = {du:8.3e}"
        )

        if k >= 2 and du < 1e-8:
            break

        # Advance the OUTER latent variable
        psi_old.vec.data = psih.vec

        # Conservative growth of the proximal parameter
        alpha_k = min(2.0 * alpha_k, 10.0)

 0: alpha = 5.000e-05, ||u^k-u^(k-1)||_L2 = 4.644e-02
 1: alpha = 1.000e-04, ||u^k-u^(k-1)||_L2 = 1.056e-03
 2: alpha = 2.000e-04, ||u^k-u^(k-1)||_L2 = 4.745e-05
 3: alpha = 4.000e-04, ||u^k-u^(k-1)||_L2 = 9.007e-07
 4: alpha = 8.000e-04, ||u^k-u^(k-1)||_L2 = 8.594e-09


In [17]:
# Latent gap:
#
#     gap = u_y - phi = exp(psi)
latent_gap = exp(psih)

# Bound-preserving vertical displacement
latent_uy = phi + latent_gap


Draw(
    gfu,
    mesh,
    "without_contact",
    deformation=True,
)

Draw(
    uh,
    mesh,
    "with_contact",
    deformation=True,
)


WebGLScene

In [18]:
import numpy as np
import matplotlib.pyplot as plt

# Avoid the corner points, because they belong to several boundaries
xs = np.linspace(1e-8, 1.0 - 1e-8, 600)

pvals = []

for xi in xs:
    mp = mesh(float(xi), 0.0, 0.0, BND)

    if mp.nr == -1:
        pvals.append(np.nan)
    else:
        pvals.append(float(pressure(mp)))

pvals = np.asarray(pvals)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.5))

ax.plot(xs, pvals, linewidth=1.5)
ax.axhline(0.0, linewidth=0.8)
ax.axvline(0.75, linestyle="--", linewidth=1.0)

ax.set_xlabel(r"$x$")
ax.set_ylabel(r"$p_h(x)$")
ax.set_title("Contact pressure on the bottom boundary")
ax.grid(True)
ax.set_xlim(0.0, 1.0)

fig.tight_layout()
plt.show()